# Digikala Recommendation Status — Transformer Encoders

این Notebook مرحلهٔ دوم آزمایش است: مقایسهٔ کنترل‌شدهٔ **ParsBERT** و **XLM-RoBERTa-base** روی همان نمونه و همان split مربوط به baseline کلاسیک. معیار اصلی انتخاب `Macro-F1` روی validation است؛ test تا بعد از انتخاب مدل استفاده نمی‌شود.

## پیش‌نیاز اجرای Kaggle

1. در `Settings > Accelerator` یک GPU انتخاب کنید. T4/L4/A100 مستقیماً اجرا می‌شوند؛ برای P100 سلول اول به‌صورت خودکار build سازگار `cu126` را نصب می‌کند.
2. گزینهٔ Internet را روشن کنید.
3. بهترین حالت این است که خروجی Notebook قبلی را با `Add Input` اضافه کنید تا فایل `sampled_split_manifest.csv` پیدا شود. در این صورت دقیقاً همان رکوردها و split استفاده می‌شوند.
4. اگر manifest اضافه نشده باشد، Notebook نمونه و split را با همان seed و منطق قبلی از دیتاست pin‌شده بازسازی می‌کند.
5. اگر همین Notebook را قبلاً اجرا کرده‌اید، ابتدا `Restart Session` بزنید؛ سپس سلول‌ها را از ابتدا به‌ترتیب اجرا کنید.

این نسخه برای CUDA نوشته شده است؛ TPU v5e بدون بازنویسی جداگانه با `torch_xla` پشتیبانی نمی‌شود. تنظیم پیش‌فرض برای حافظهٔ 16GB محافظه‌کارانه است: batch برابر 8 و gradient accumulation برابر 4. اگر اجرای هر دو مدل طولانی شد، فقط برای smoke test مقدار `CANDIDATE_EPOCHS` را 1 کنید؛ برای نتیجهٔ قابل گزارش مقدار پیش‌فرض 2 را نگه دارید.

In [ ]:
from __future__ import annotations

import subprocess
import sys

# این سلول باید در یک session تازه و پیش از import کردن torch/transformers اجرا شود.
def subprocess_text(command):
    return subprocess.check_output(command, stderr=subprocess.DEVNULL, text=True).strip()

try:
    gpu_line = subprocess_text([
        'nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader,nounits'
    ]).splitlines()[0]
    gpu_name_before_import, capability_text = [part.strip() for part in gpu_line.rsplit(',', 1)]
    gpu_compute_capability = float(capability_text)
except Exception:
    gpu_name_before_import = 'unknown'
    gpu_compute_capability = None

try:
    installed_torch_cuda = subprocess_text([
        sys.executable, '-c', 'import torch; print(torch.version.cuda or \"cpu\")'
    ])
except Exception:
    installed_torch_cuda = 'missing'

legacy_gpu = (
    gpu_compute_capability is not None and gpu_compute_capability < 7.5
) or ('P100' in gpu_name_before_import or 'V100' in gpu_name_before_import)
needs_cu126 = legacy_gpu and not installed_torch_cuda.startswith('12.6')
torch_was_already_imported = 'torch' in sys.modules

print({
    'detected_gpu': gpu_name_before_import,
    'compute_capability': gpu_compute_capability,
    'installed_torch_cuda': installed_torch_cuda,
    'needs_cu126': needs_cu126,
})

if needs_cu126:
    print('Installing the PyTorch 2.10 CUDA 12.6 build required by Pascal/Volta GPUs ...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
        'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    if torch_was_already_imported:
        raise RuntimeError(
            'PyTorch cu126 نصب شد، اما نسخهٔ قبلی هنوز در حافظه است. ' 
            'اکنون Restart Session بزنید و Notebook را دوباره از سلول اول اجرا کنید.'
        )

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48,<5',
    'accelerate>=1.2,<2',
    'sentencepiece>=0.2',
    'safetensors>=0.4',
])

torch_build = subprocess_text([
    sys.executable, '-c',
    'import torch; print(torch.__version__, torch.version.cuda, \",\".join(torch.cuda.get_arch_list()), sep=\"|\")',
])
print('Runtime build:', torch_build)
print('Transformer dependencies are ready.')

In [ ]:
import gc
import hashlib
import inspect
import json
import math
import os
import platform
import random
import socket
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
VALID_LABELS = ['recommended', 'not_recommended', 'no_idea']
LABEL2ID = {label: index for index, label in enumerate(VALID_LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}

HF_REPO_ID = 'RadeAI/Digikala_comments_products'
HF_REVISION = '89c3133b169c8d3793db8834f56f32fee33d9db0'
HF_FILENAME = 'digikala-comments.csv'
HF_EXPECTED_SIZE = 1_278_526_959
HF_EXPECTED_SHA256 = 'c7a8aa3020334fde8ec24944576a03fe5785e6fe12cd01042f5836632ddf8297'
HF_DOWNLOAD_URL = f'https://huggingface.co/datasets/{HF_REPO_ID}/resolve/{HF_REVISION}/{HF_FILENAME}?download=true'

COMMENTS_PATH = os.getenv('DIGIKALA_COMMENTS_PATH') or None
MANIFEST_PATH = os.getenv('DIGIKALA_MANIFEST_PATH') or None
SAMPLE_FRACTION = 0.02
MAX_SAMPLED_ROWS = 150_000
CHUNK_SIZE = 250_000

MODEL_SPECS = [
    {'name': 'parsbert', 'checkpoint': 'HooshvareLab/bert-fa-base-uncased'},
    {'name': 'xlm_roberta_base', 'checkpoint': 'FacebookAI/xlm-roberta-base'},
]
CANDIDATE_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_LENGTH_CAP = 160
TOKEN_LENGTH_PROBE_ROWS = 20_000

BASELINE_VALIDATION_MACRO_F1 = 0.6806198033446752
BASELINE_TEST_MACRO_F1_REFERENCE = 0.6611412090831573
MIN_ABSOLUTE_VALIDATION_GAIN = 0.02
EXPECTED_SPLIT_PROFILE = {
    'train': {'rows': 85_694, 'text_groups': 66_899, 'recommended': 68_106, 'not_recommended': 8_159, 'no_idea': 9_429},
    'validation': {'rows': 9_941, 'text_groups': 8_249, 'recommended': 7_678, 'not_recommended': 993, 'no_idea': 1_270},
    'test': {'rows': 9_662, 'text_groups': 8_215, 'recommended': 7_612, 'not_recommended': 980, 'no_idea': 1_070},
}

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'outputs' / 'kaggle_transformers'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_DIR / 'transformer_candidate_runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('GPU پیدا نشد. در Kaggle از Settings > Accelerator یک GPU انتخاب و session را restart کنید.')

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = tuple(torch.cuda.get_device_capability(0))
compiled_cuda_arches = torch.cuda.get_arch_list()
device_arch = f'sm_{gpu_capability[0]}{gpu_capability[1]}'
if device_arch not in compiled_cuda_arches:
    raise RuntimeError(
        f'GPU {gpu_name} requires {device_arch}, but this PyTorch build supports {compiled_cuda_arches}. '
        'For P100/V100 restart the session and run the compatibility cell first.'
    )

# یک kernel واقعی اجرا می‌شود تا ناسازگاری CUDA همین‌جا و با پیام روشن مشخص شود.
cuda_probe = torch.ones(8, device='cuda').sum()
torch.cuda.synchronize()
del cuda_probe

use_bf16 = gpu_capability[0] >= 8 and bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
use_fp16 = not use_bf16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

print(json.dumps({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'scikit_learn': sklearn.__version__,
    'gpu': gpu_name,
    'gpu_compute_capability': gpu_capability,
    'torch_cuda_runtime': torch.version.cuda,
    'compiled_cuda_arches': compiled_cuda_arches,
    'precision': 'bf16' if use_bf16 else 'fp16',
    'output_dir': str(OUTPUT_DIR),
}, ensure_ascii=False, indent=2))

## دریافت و اعتبارسنجی منبع داده

فایل CSV دقیقاً از revision مشخص Hugging Face دریافت می‌شود و اندازه و SHA-256 آن کنترل می‌شود. بنابراین تغییرات بعدی repository روی این آزمایش اثر ندارند.

In [ ]:
def file_sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def validate_hf_file(path: Path, verify_hash: bool = True) -> Path:
    actual_size = path.stat().st_size
    if actual_size != HF_EXPECTED_SIZE:
        raise ValueError(f'Unexpected file size: {actual_size:,}; expected {HF_EXPECTED_SIZE:,}')
    if verify_hash:
        actual_hash = file_sha256(path)
        if actual_hash != HF_EXPECTED_SHA256:
            raise ValueError(f'SHA256 mismatch: {actual_hash}')
    return path

def assert_huggingface_network() -> None:
    try:
        socket.getaddrinfo('huggingface.co', 443, type=socket.SOCK_STREAM)
    except socket.gaierror as error:
        raise RuntimeError(
            'Kaggle Internet در دسترس نیست. Internet را روشن، session را restart و Notebook را از ابتدا اجرا کنید.'
        ) from error

def direct_download_from_hf(target: Path) -> Path:
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_suffix(target.suffix + '.part')
    request = urllib.request.Request(HF_DOWNLOAD_URL, headers={'User-Agent': 'kaggle-digikala-transformer/1.0'})
    print('Direct download:', HF_DOWNLOAD_URL)
    downloaded = 0
    report_step = 128 * 1024 * 1024
    next_report = report_step
    with urllib.request.urlopen(request, timeout=120) as response, partial.open('wb') as output:
        while block := response.read(8 * 1024 * 1024):
            output.write(block)
            downloaded += len(block)
            if downloaded >= next_report:
                print(f'Downloaded: {downloaded / 1_000_000_000:.2f} GB')
                next_report += report_step
    partial.replace(target)
    return target

def get_comments_csv(manual_path: str | None = None) -> Path:
    if manual_path:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f'COMMENTS_PATH does not exist: {path}')
        print('Using manual/local CSV override.')
        return validate_hf_file(path)

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        input_candidates = [
            path for path in kaggle_input.rglob(HF_FILENAME)
            if path.is_file() and path.stat().st_size == HF_EXPECTED_SIZE
        ]
        for candidate in input_candidates:
            try:
                print('Validating dataset already attached as Kaggle Input:', candidate)
                return validate_hf_file(candidate)
            except ValueError:
                pass

    direct_target = OUTPUT_DIR / 'hf_data' / HF_FILENAME
    if direct_target.exists():
        try:
            return validate_hf_file(direct_target)
        except ValueError as error:
            print('Cached direct file is invalid; downloading again:', error)

    assert_huggingface_network()
    try:
        from huggingface_hub import hf_hub_download
        cached_path = Path(hf_hub_download(
            repo_id=HF_REPO_ID,
            repo_type='dataset',
            filename=HF_FILENAME,
            revision=HF_REVISION,
            cache_dir=str(OUTPUT_DIR / 'hf_cache'),
        ))
        return validate_hf_file(cached_path)
    except Exception as error:
        print(f'huggingface_hub failed ({type(error).__name__}: {error}); trying direct URL.')
        assert_huggingface_network()
        return validate_hf_file(direct_download_from_hf(direct_target))

comments_path = get_comments_csv(COMMENTS_PATH)
print('Comments CSV:', comments_path)
print('Verified size (GB):', round(comments_path.stat().st_size / 1_000_000_000, 3))
print('Verified SHA256:', HF_EXPECTED_SHA256)

## بازیابی همان نمونه و split

اول manifest اجرای baseline جست‌وجو می‌شود. اگر پیدا شود فقط شناسه‌های همان فایل از CSV خوانده می‌شوند. در fallback، منطق نمونه‌گیری، پاک‌سازی متن، group key و `StratifiedGroupKFold` دقیقاً مطابق Notebook اول است. هیچ ویژگیِ target داخل متن وارد نمی‌شود.

In [ ]:
TEXT_COLUMNS = ['id', 'title', 'body', 'advantages', 'disadvantages', 'recommendation_status', 'product_id']
NULL_TOKENS = {'', 'nan', 'none', 'null', 'na', 'n/a'}
ARABIC_TO_PERSIAN = str.maketrans({'ي': 'ی', 'ى': 'ی', 'ك': 'ک'})

def normalize_text_series(series: pd.Series) -> pd.Series:
    out = series.fillna('').astype(str)
    stripped_lower = out.str.strip().str.lower()
    out = out.mask(stripped_lower.isin(NULL_TOKENS), '')
    out = out.str.normalize('NFKC').str.translate(ARABIC_TO_PERSIAN)
    out = out.str.replace('\ufeff', '', regex=False)
    out = out.str.replace(r'\s+', ' ', regex=True).str.strip()
    return out

def stable_sample_mask(ids: pd.Series, fraction: float, seed: int = 42) -> np.ndarray:
    sample_keys = ids.astype(str) + f'-{seed}'
    hashed = pd.util.hash_pandas_object(sample_keys, index=False).to_numpy(dtype=np.uint64)
    scale = np.uint64(1_000_000)
    threshold = int(round(fraction * int(scale)))
    return (hashed % scale) < threshold

def build_model_text(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for column in ['title', 'body', 'advantages', 'disadvantages']:
        frame[column] = normalize_text_series(frame[column])
    tagged_parts = []
    for column, tag in [('title', '[TITLE]'), ('body', '[BODY]'), ('advantages', '[ADVANTAGES]'), ('disadvantages', '[DISADVANTAGES]')]:
        tagged_parts.append(np.where(frame[column].ne(''), tag + ' ' + frame[column] + ' ', ''))
    full_text = pd.Series(tagged_parts[0], index=frame.index)
    for part in tagged_parts[1:]:
        full_text = full_text + pd.Series(part, index=frame.index)
    frame['text_full'] = full_text.str.replace(r'\s+', ' ', regex=True).str.strip()
    frame['text_body'] = frame['body']
    split_text = frame['text_body'].where(frame['text_body'].ne(''), frame['text_full'])
    frame['text_group_id'] = split_text.map(lambda value: hashlib.sha1(value.encode('utf-8')).hexdigest())
    return frame

def cap_sample_by_complete_groups(frame: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    if max_rows <= 0 or len(frame) <= max_rows:
        return frame
    group_sizes = frame.groupby('text_group_id', sort=False).size().rename('rows').reset_index()
    order_hash = pd.util.hash_pandas_object(group_sizes['text_group_id'] + f'-{seed}', index=False).to_numpy(dtype=np.uint64)
    group_sizes = group_sizes.assign(order_hash=order_hash).sort_values('order_hash')
    selected = group_sizes.loc[group_sizes['rows'].cumsum() <= max_rows, 'text_group_id']
    if selected.empty:
        selected = group_sizes.head(1)['text_group_id']
    return frame[frame['text_group_id'].isin(set(selected))].copy()

def create_splits(frame: pd.DataFrame, seed: int = 42) -> dict[str, pd.DataFrame]:
    y = frame['recommendation_status'].to_numpy()
    groups = frame['text_group_id'].to_numpy()
    x = np.zeros(len(frame), dtype=np.uint8)
    outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
    train_val_idx, test_idx = next(outer.split(x, y, groups))
    train_val = frame.iloc[train_val_idx].copy()
    inner = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=seed + 1)
    train_rel_idx, val_rel_idx = next(inner.split(
        np.zeros(len(train_val), dtype=np.uint8),
        train_val['recommendation_status'].to_numpy(),
        train_val['text_group_id'].to_numpy(),
    ))
    return {
        'train': train_val.iloc[train_rel_idx].copy(),
        'validation': train_val.iloc[val_rel_idx].copy(),
        'test': frame.iloc[test_idx].copy(),
    }

def find_manifest(manual_path: str | None = None) -> Path | None:
    if manual_path:
        path = Path(manual_path)
        if not path.exists():
            raise FileNotFoundError(f'MANIFEST_PATH does not exist: {path}')
        return path
    candidates = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]:
        if root.exists():
            candidates.extend(root.rglob('sampled_split_manifest.csv'))
    candidates = [path for path in candidates if path.resolve() != (OUTPUT_DIR / 'sampled_split_manifest.csv').resolve()]
    return sorted(candidates, key=lambda path: len(str(path)))[0] if candidates else None

def rows_for_manifest(csv_path: Path, manifest: pd.DataFrame) -> pd.DataFrame:
    wanted = set(manifest['id'].astype(str))
    chunks = []
    for chunk_number, chunk in enumerate(pd.read_csv(
        csv_path, usecols=TEXT_COLUMNS, dtype=str, chunksize=CHUNK_SIZE,
        keep_default_na=False, na_filter=False, encoding='utf-8-sig',
    ), start=1):
        chosen = chunk[chunk['id'].astype(str).isin(wanted)].copy()
        if not chosen.empty:
            chunks.append(chosen)
        if chunk_number % 5 == 0:
            print(f'Chunks: {chunk_number:,} | recovered rows: {sum(map(len, chunks)):,}/{len(wanted):,}')
    frame = pd.concat(chunks, ignore_index=True)
    frame = frame.drop_duplicates('id', keep='first')
    frame = build_model_text(frame)
    merged = manifest.merge(
        frame.drop(columns=['product_id', 'recommendation_status', 'text_group_id'], errors='ignore'),
        on='id', how='left', validate='one_to_one',
    )
    missing = int(merged['text_full'].isna().sum())
    if missing:
        raise RuntimeError(f'{missing} manifest ids were not recovered from the verified source CSV.')
    return merged

def rebuild_sample(csv_path: Path) -> tuple[pd.DataFrame, dict]:
    sampled_chunks = []
    scan_rows = 0
    valid_label_rows = 0
    for chunk_number, chunk in enumerate(pd.read_csv(
        csv_path, usecols=TEXT_COLUMNS, dtype=str, chunksize=CHUNK_SIZE,
        keep_default_na=False, na_filter=False, encoding='utf-8-sig',
    ), start=1):
        scan_rows += len(chunk)
        chunk['recommendation_status'] = chunk['recommendation_status'].astype(str).str.strip()
        chunk = chunk[chunk['recommendation_status'].isin(VALID_LABELS)].copy()
        valid_label_rows += len(chunk)
        if not chunk.empty:
            chunk = chunk.loc[stable_sample_mask(chunk['id'], SAMPLE_FRACTION, SEED)].copy()
            if not chunk.empty:
                sampled_chunks.append(chunk)
        if chunk_number % 5 == 0:
            print(f'Chunks: {chunk_number:,} | scanned: {scan_rows:,} | sampled: {sum(map(len, sampled_chunks)):,}')
    frame = pd.concat(sampled_chunks, ignore_index=True)
    duplicate_ids = int(frame.duplicated('id', keep='first').sum())
    frame = build_model_text(frame).drop_duplicates('id', keep='first')
    empty_texts = int(frame['text_full'].eq('').sum())
    frame = frame[frame['text_full'].ne('')].copy()
    frame = cap_sample_by_complete_groups(frame, MAX_SAMPLED_ROWS, SEED).reset_index(drop=True)
    return frame, {
        'physical_rows_scanned': int(scan_rows),
        'valid_label_rows': int(valid_label_rows),
        'duplicate_comment_id_rows_removed': duplicate_ids,
        'empty_text_rows_removed': empty_texts,
    }

In [ ]:
manifest_source = find_manifest(MANIFEST_PATH)
rebuild_audit = {}

if manifest_source is not None:
    print('Using previous baseline manifest:', manifest_source)
    manifest = pd.read_csv(manifest_source, dtype=str, keep_default_na=False)
    required = {'id', 'product_id', 'text_group_id', 'recommendation_status', 'split'}
    if not required.issubset(manifest.columns):
        raise ValueError(f'Manifest is missing columns: {sorted(required - set(manifest.columns))}')
    if manifest['id'].duplicated().any():
        raise ValueError('Manifest contains duplicate comment ids.')
    if not set(manifest['split']).issubset({'train', 'validation', 'test'}):
        raise ValueError('Manifest contains an unknown split name.')
    if not set(manifest['recommendation_status']).issubset(VALID_LABELS):
        raise ValueError('Manifest contains an unknown target label.')
    sample = rows_for_manifest(comments_path, manifest)
    split_frames = {name: sample[sample['split'].eq(name)].copy() for name in ['train', 'validation', 'test']}
    split_source = 'previous_baseline_manifest'
else:
    print('Manifest not found; deterministically rebuilding the sample and split.')
    sample, rebuild_audit = rebuild_sample(comments_path)
    split_frames = create_splits(sample, SEED)
    split_source = 'deterministic_rebuild'

train_df = split_frames['train'].reset_index(drop=True)
val_df = split_frames['validation'].reset_index(drop=True)
test_df = split_frames['test'].reset_index(drop=True)

for frame in [train_df, val_df, test_df]:
    frame['label_id'] = frame['recommendation_status'].map(LABEL2ID).astype(int)

for left, right in [('train', 'validation'), ('train', 'test'), ('validation', 'test')]:
    assert set(split_frames[left]['id']).isdisjoint(split_frames[right]['id'])
    assert set(split_frames[left]['text_group_id']).isdisjoint(split_frames[right]['text_group_id'])

split_rows = []
for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    counts = frame['recommendation_status'].value_counts()
    split_rows.append({
        'split': name,
        'rows': int(len(frame)),
        'text_groups': int(frame['text_group_id'].nunique()),
        **{label: int(counts.get(label, 0)) for label in VALID_LABELS},
    })
split_profile = pd.DataFrame(split_rows)
for row in split_rows:
    expected = EXPECTED_SPLIT_PROFILE[row['split']]
    observed = {key: row[key] for key in expected}
    if observed != expected:
        raise RuntimeError(
            f"Split {row['split']} does not match the baseline run. "
            f'Observed={observed}, expected={expected}. Attach the original sampled_split_manifest.csv.'
        )
display(split_profile)
print('Split source:', split_source)
print('Exact baseline profile and leakage checks passed.')

manifest_parts = []
for split_name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    part = frame[['id', 'product_id', 'text_group_id', 'recommendation_status']].copy()
    part['split'] = split_name
    manifest_parts.append(part)
manifest_output_path = OUTPUT_DIR / 'sampled_split_manifest.csv'
pd.concat(manifest_parts, ignore_index=True).to_csv(manifest_output_path, index=False)
print('Saved active manifest:', manifest_output_path)

## Dataset، معیارها و Trainer وزن‌دار

برای عدم توازن کلاس‌ها از cross-entropy وزن‌دار بر اساس فراوانی train استفاده می‌شود. طول ورودی برای هر tokenizer جداگانه از صدک 99 یک probe محاسبه و حداکثر روی 160 token محدود می‌شود. padding به‌صورت پویا در هر batch انجام می‌شود.

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(map(int, labels))
        self.tokenizer = tokenizer
        self.max_length = int(max_length)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=self.max_length,
            padding=False,
        )
        item['labels'] = self.labels[index]
        return item

def calculate_metrics(y_true_ids, y_pred_ids) -> dict:
    y_true_ids = np.asarray(y_true_ids)
    y_pred_ids = np.asarray(y_pred_ids)
    result = {
        'macro_f1': float(f1_score(y_true_ids, y_pred_ids, labels=list(ID2LABEL), average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true_ids, y_pred_ids, labels=list(ID2LABEL), average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true_ids, y_pred_ids)),
    }
    per_class = classification_report(
        y_true_ids, y_pred_ids, labels=list(ID2LABEL), output_dict=True, zero_division=0
    )
    for label_id, label_name in ID2LABEL.items():
        row = per_class[str(label_id)]
        result[f'precision_{label_name}'] = float(row['precision'])
        result[f'recall_{label_name}'] = float(row['recall'])
        result[f'f1_{label_name}'] = float(row['f1-score'])
    return result

def trainer_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return calculate_metrics(labels, predictions)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device)
        loss = nn.CrossEntropyLoss(weight=weights)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def class_weights_for(frame: pd.DataFrame) -> list[float]:
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(len(VALID_LABELS)),
        y=frame['label_id'].to_numpy(),
    )
    return [float(value) for value in weights]

def estimate_max_length(tokenizer, texts: pd.Series) -> tuple[int, dict]:
    probe = texts.sample(min(TOKEN_LENGTH_PROBE_ROWS, len(texts)), random_state=SEED).tolist()
    encoded = tokenizer(probe, truncation=False, padding=False, return_length=True)
    lengths = np.asarray(encoded['length'], dtype=np.int32)
    stats = {
        'probe_rows': int(len(lengths)),
        'p50': float(np.quantile(lengths, 0.50)),
        'p90': float(np.quantile(lengths, 0.90)),
        'p95': float(np.quantile(lengths, 0.95)),
        'p99': float(np.quantile(lengths, 0.99)),
        'max': int(lengths.max()),
    }
    chosen = max(64, min(MAX_LENGTH_CAP, int(math.ceil(stats['p99'] / 8) * 8)))
    return chosen, stats

def make_training_arguments(output_dir: Path, epochs: int, do_eval: bool) -> TrainingArguments:
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=int(epochs),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type='linear',
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=100,
        report_to='none',
        seed=SEED,
        data_seed=SEED,
        dataloader_num_workers=2,
        save_total_limit=1,
        remove_unused_columns=True,
    )
    signature = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = 'eval_strategy' if 'eval_strategy' in signature else 'evaluation_strategy'
    if do_eval:
        kwargs.update({
            eval_key: 'epoch',
            'save_strategy': 'epoch',
            'load_best_model_at_end': True,
            'metric_for_best_model': 'macro_f1',
            'greater_is_better': True,
        })
    else:
        kwargs.update({eval_key: 'no', 'save_strategy': 'no'})
    return TrainingArguments(**kwargs)

def make_trainer(model, tokenizer, train_dataset, eval_dataset, output_dir, epochs, class_weights):
    arguments = make_training_arguments(output_dir, epochs, eval_dataset is not None)
    kwargs = dict(
        model=model,
        args=arguments,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer,
            pad_to_multiple_of=8,
            return_tensors='pt',
        ),
        compute_metrics=trainer_metrics if eval_dataset is not None else None,
        class_weights=class_weights,
    )
    trainer_signature = inspect.signature(Trainer.__init__).parameters
    if 'processing_class' in trainer_signature:
        kwargs['processing_class'] = tokenizer
    else:
        kwargs['tokenizer'] = tokenizer
    return WeightedTrainer(**kwargs)

## آموزش و مقایسهٔ candidateها روی validation

هر checkpoint از ابتدا fine-tune می‌شود. انتخاب فقط با `validation Macro-F1` انجام می‌شود و علاوه بر معیار کلی، F1 هر کلاس—به‌خصوص `no_idea`—ثبت می‌شود. ممکن است این سلول چند ساعت زمان ببرد.

In [ ]:
candidate_results = []
train_weights = class_weights_for(train_df)
print('Balanced class weights:', dict(zip(VALID_LABELS, train_weights)))

for spec in MODEL_SPECS:
    print('\n' + '=' * 90)
    print('Training candidate:', spec['name'], '|', spec['checkpoint'])
    print('=' * 90)
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(spec['checkpoint'], use_fast=True)
    max_length, length_stats = estimate_max_length(tokenizer, train_df['text_full'])
    print('Token length statistics:', length_stats)
    print('Selected max_length:', max_length)

    train_dataset = TextClassificationDataset(
        train_df['text_full'], train_df['label_id'], tokenizer, max_length
    )
    validation_dataset = TextClassificationDataset(
        val_df['text_full'], val_df['label_id'], tokenizer, max_length
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        spec['checkpoint'],
        num_labels=len(VALID_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    resolved_revision = getattr(model.config, '_commit_hash', None)
    parameter_count = int(sum(parameter.numel() for parameter in model.parameters()))
    run_dir = RUNS_DIR / spec['name']
    trainer = make_trainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_dataset, eval_dataset=validation_dataset,
        output_dir=run_dir, epochs=CANDIDATE_EPOCHS, class_weights=train_weights,
    )

    fit_start = time.perf_counter()
    trainer.train()
    fit_seconds = time.perf_counter() - fit_start
    predict_start = time.perf_counter()
    validation_output = trainer.predict(validation_dataset)
    predict_seconds = time.perf_counter() - predict_start
    validation_prediction = np.argmax(validation_output.predictions, axis=-1)
    metrics = calculate_metrics(val_df['label_id'], validation_prediction)

    eval_logs = [row for row in trainer.state.log_history if 'eval_macro_f1' in row]
    best_log = max(eval_logs, key=lambda row: row['eval_macro_f1']) if eval_logs else {'epoch': CANDIDATE_EPOCHS}
    best_epoch = max(1, int(round(float(best_log.get('epoch', CANDIDATE_EPOCHS)))))
    result = {
        'model': spec['name'],
        'checkpoint': spec['checkpoint'],
        'resolved_revision': resolved_revision,
        'parameter_count': parameter_count,
        'max_length': int(max_length),
        'best_epoch': best_epoch,
        'fit_seconds': float(fit_seconds),
        'validation_predict_seconds': float(predict_seconds),
        **metrics,
        'token_length_stats': length_stats,
    }
    candidate_results.append(result)
    display(pd.DataFrame([{key: value for key, value in result.items() if key != 'token_length_stats'}]))

    del trainer, model, tokenizer, train_dataset, validation_dataset, validation_output, validation_prediction
    gc.collect()
    torch.cuda.empty_cache()

validation_leaderboard = pd.DataFrame([
    {key: value for key, value in row.items() if key != 'token_length_stats'}
    for row in candidate_results
]).sort_values('macro_f1', ascending=False).reset_index(drop=True)
display(validation_leaderboard)

## Promotion gate و ارزیابی نهایی

مدل فقط وقتی به مرحلهٔ test می‌رود که `Macro-F1` اعتبارسنجی آن دست‌کم 0.02 از baseline بهتر باشد. اگر این شرط برقرار نباشد، سلول بدون دست‌زدن به test متوقف نمی‌شود؛ نتیجه را ذخیره می‌کند و پیشنهاد می‌دهد baseline کلاسیک حفظ شود. اگر شرط برقرار باشد، checkpoint برنده با تعداد epoch برگزیده روی train+validation دوباره آموزش می‌بیند و test دقیقاً یک بار ارزیابی می‌شود.

In [ ]:
best_candidate = max(candidate_results, key=lambda row: row['macro_f1'])
validation_gain = float(best_candidate['macro_f1'] - BASELINE_VALIDATION_MACRO_F1)
promoted = bool(validation_gain >= MIN_ABSOLUTE_VALIDATION_GAIN)

print('Selected candidate:', best_candidate['model'])
print(f"Validation Macro-F1: {best_candidate['macro_f1']:.4f}")
print(f'Baseline Validation Macro-F1: {BASELINE_VALIDATION_MACRO_F1:.4f}')
print(f'Absolute gain: {validation_gain:+.4f}')
print('Promotion threshold:', MIN_ABSOLUTE_VALIDATION_GAIN)
print('Promoted to final test:', promoted)

test_metrics = None
test_per_class = None
test_confusion_matrix = None
final_fit_seconds = None
test_predict_seconds = None
final_model_dir = OUTPUT_DIR / 'best_transformer_encoder'

if promoted:
    set_seed(SEED)
    train_val_df = pd.concat([train_df, val_df], ignore_index=True)
    tokenizer = AutoTokenizer.from_pretrained(best_candidate['checkpoint'], use_fast=True)
    max_length = int(best_candidate['max_length'])
    train_val_dataset = TextClassificationDataset(
        train_val_df['text_full'], train_val_df['label_id'], tokenizer, max_length
    )
    test_dataset = TextClassificationDataset(
        test_df['text_full'], test_df['label_id'], tokenizer, max_length
    )
    final_model = AutoModelForSequenceClassification.from_pretrained(
        best_candidate['checkpoint'],
        num_labels=len(VALID_LABELS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    final_trainer = make_trainer(
        model=final_model, tokenizer=tokenizer,
        train_dataset=train_val_dataset, eval_dataset=None,
        output_dir=OUTPUT_DIR / 'final_training_run',
        epochs=int(best_candidate['best_epoch']),
        class_weights=class_weights_for(train_val_df),
    )
    fit_start = time.perf_counter()
    final_trainer.train()
    final_fit_seconds = float(time.perf_counter() - fit_start)

    predict_start = time.perf_counter()
    test_output = final_trainer.predict(test_dataset)
    test_predict_seconds = float(time.perf_counter() - predict_start)
    test_prediction = np.argmax(test_output.predictions, axis=-1)
    test_metrics = calculate_metrics(test_df['label_id'], test_prediction)
    report = classification_report(
        test_df['label_id'], test_prediction, labels=list(ID2LABEL), output_dict=True, zero_division=0
    )
    test_per_class = {ID2LABEL[index]: report[str(index)] for index in ID2LABEL}
    test_confusion_matrix = confusion_matrix(
        test_df['label_id'], test_prediction, labels=list(ID2LABEL)
    ).tolist()

    final_model_dir.mkdir(parents=True, exist_ok=True)
    final_trainer.save_model(str(final_model_dir))
    tokenizer.save_pretrained(str(final_model_dir))
    (final_model_dir / 'inference_config.json').write_text(json.dumps({
        'max_length': max_length,
        'labels': VALID_LABELS,
        'normalization_version': 'fa_light_v1',
        'text_column': 'text_full',
    }, ensure_ascii=False, indent=2), encoding='utf-8')

    print('FINAL TEST metrics:')
    print(json.dumps(test_metrics, ensure_ascii=False, indent=2))
    print('Per-class report:')
    print(json.dumps(test_per_class, ensure_ascii=False, indent=2))
else:
    print('مدل Transformer بهبود لازم را نداشت؛ test ارزیابی نشد و baseline کلاسیک فعلاً انتخاب بهتر است.')

In [ ]:
validation_results_path = OUTPUT_DIR / 'transformer_validation_results.csv'
summary_path = OUTPUT_DIR / 'transformer_run_summary.json'
validation_leaderboard.to_csv(validation_results_path, index=False)

summary = {
    'task': 'digikala_recommendation_status_transformer_encoders',
    'seed': SEED,
    'huggingface_repo': HF_REPO_ID,
    'huggingface_revision': HF_REVISION,
    'huggingface_filename': HF_FILENAME,
    'source_sha256': HF_EXPECTED_SHA256,
    'source_file': str(comments_path),
    'source_size_bytes': int(comments_path.stat().st_size),
    'split_source': split_source,
    'manifest_source': str(manifest_source) if manifest_source else None,
    'sample_fraction': SAMPLE_FRACTION,
    'max_sampled_rows': MAX_SAMPLED_ROWS,
    'split_profile': split_profile.to_dict(orient='records'),
    'rebuild_audit': rebuild_audit,
    'training_config': {
        'candidate_epochs': CANDIDATE_EPOCHS,
        'train_batch_size': TRAIN_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'effective_train_batch_size': TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'warmup_ratio': WARMUP_RATIO,
        'max_length_cap': MAX_LENGTH_CAP,
        'class_weighting': 'balanced_train_frequency',
    },
    'baseline_validation_macro_f1': BASELINE_VALIDATION_MACRO_F1,
    'baseline_test_macro_f1_reference_only': BASELINE_TEST_MACRO_F1_REFERENCE,
    'minimum_absolute_validation_gain': MIN_ABSOLUTE_VALIDATION_GAIN,
    'validation_candidates': candidate_results,
    'selected_model': best_candidate['model'],
    'selected_checkpoint': best_candidate['checkpoint'],
    'selected_resolved_revision': best_candidate['resolved_revision'],
    'selected_validation_macro_f1': best_candidate['macro_f1'],
    'absolute_validation_gain': validation_gain,
    'promoted_to_test': promoted,
    'final_fit_seconds': final_fit_seconds,
    'test_predict_seconds': test_predict_seconds,
    'test_metrics': test_metrics,
    'test_per_class': test_per_class,
    'test_confusion_matrix_label_order': VALID_LABELS if promoted else None,
    'test_confusion_matrix': test_confusion_matrix,
    'final_model_dir': str(final_model_dir) if promoted else None,
    'versions': {
        'python': sys.version.split()[0],
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'platform': platform.platform(),
        'gpu': gpu_name,
        'gpu_compute_capability': list(gpu_capability),
        'torch_cuda_runtime': torch.version.cuda,
        'compiled_cuda_arches': compiled_cuda_arches,
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved artifacts:')
for path in [validation_results_path, manifest_output_path, summary_path]:
    print(f' - {path} ({path.stat().st_size / 1_000_000:.2f} MB)')
if promoted:
    model_size = sum(path.stat().st_size for path in final_model_dir.rglob('*') if path.is_file())
    print(f' - {final_model_dir} ({model_size / 1_000_000:.2f} MB)')

print('\n' + '#' * 28 + ' COPY THIS SUMMARY ' + '#' * 28)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## تست دستی مدل منتخب (فقط در صورت promotion)

In [ ]:
def prepare_one_text(title='', body='', advantages='', disadvantages=''):
    frame = pd.DataFrame([{
        'title': title, 'body': body,
        'advantages': advantages, 'disadvantages': disadvantages,
    }])
    return build_model_text(frame).iloc[0]['text_full']

manual_examples = [
    {'title': 'عالی بود', 'body': 'کیفیتش خیلی خوبه و دوباره می‌خرم'},
    {'title': 'نخرید', 'body': 'کیفیت خیلی بدی داشت و مرجوعش کردم'},
    {'title': 'معمولی', 'body': 'نسبت به قیمت بد نیست ولی انتظار بیشتری داشتم'},
]

if promoted:
    manual_texts = [prepare_one_text(**example) for example in manual_examples]
    encoded = tokenizer(
        manual_texts, truncation=True, max_length=int(best_candidate['max_length']),
        padding=True, return_tensors='pt',
    ).to(final_model.device)
    final_model.eval()
    with torch.no_grad():
        logits = final_model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1).cpu().numpy()
    rows = []
    for example, probs in zip(manual_examples, probabilities):
        rows.append({
            **example,
            'prediction': ID2LABEL[int(probs.argmax())],
            **{f'p_{label}': float(probs[index]) for index, label in ID2LABEL.items()},
        })
    display(pd.DataFrame(rows))
else:
    print('مدلی به test و ذخیرهٔ نهایی promote نشده است.')

## خروجی‌ای که باید برگردانید

بعد از پایان اجرا، بلوک کامل `COPY THIS SUMMARY` را بفرستید. همچنین این فایل‌ها را از بخش Output نگه دارید:

- `transformer_run_summary.json`
- `transformer_validation_results.csv`
- `sampled_split_manifest.csv`
- پوشهٔ `best_transformer_encoder` فقط اگر `promoted_to_test=true` بود

اگر خطای CUDA out of memory رخ داد، session را restart کنید، `TRAIN_BATCH_SIZE=4` و `GRADIENT_ACCUMULATION_STEPS=8` بگذارید و از ابتدا اجرا کنید؛ effective batch همچنان 32 می‌ماند.